<div dir="rtl">

# NB02-M01 — Quran Canonical Inventory

## هدف
ثبت هویت، Hash، ساختار، شماره‌گذاری و Evidence منبع برای فایل عربی موجود بدون تغییر حتی یک حرف از متن آیات.

### PASS
6236 رکورد، 114 سوره، ترتیب دقیق آیات، کلید تکراری صفر، متن خالی صفر و Evidence منبع/حقوق موجود باشد.

### خارج از Scope
Normalization، ترجمه، Embedding و تفسیر.

</div>

In [ ]:
# NB02-C01 — Imports and project root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / ".git").exists()
        and (p / "notebooks").exists()
        and (p / "data").exists()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, write_json_atomic
from qrag_phase1.quran import parse_quran_pipe_file, sha256_file, validate_quran_records

git_info = collect_git_info(project_root)
assert git_info["branch"] == "qrag-phase1"
print("Root:", project_root)

<div dir="rtl">

# NB02-M02 — Load Source Contract

Contract شماره آیات و Source Evidence رسمی Tanzil خوانده می‌شوند. Evidence وب، جای Structural validation فایل محلی را نمی‌گیرد.

</div>

In [ ]:
# NB02-C02 — Load canonical contract and source evidence
contract_path = project_root / "config/phase1/quran_canonical_contract_v1.json"
evidence_path = project_root / "data/phase1/source_evidence/quran_tanzil_source_evidence_v1.json"
raw_path = project_root / "data/raw/quran_arabic_uthmani.txt"

for path in (contract_path, evidence_path, raw_path):
    assert path.is_file(), f"Missing required input: {path}"

contract = json.loads(contract_path.read_text(encoding="utf-8"))
source_evidence = json.loads(evidence_path.read_text(encoding="utf-8"))

assert contract["expected_verse_count"] == 6236
assert len(contract["expected_verse_counts_by_surah"]) == 114
assert sum(contract["expected_verse_counts_by_surah"]) == 6236
assert source_evidence["source_id"] == contract["source_id"]

<div dir="rtl">

# NB02-M03 — Parse Raw File

Headerها جدا نگه داشته می‌شوند و Verse text بدون Normalization خوانده می‌شود.

</div>

In [ ]:
# NB02-C03 — Parse raw Uthmani file
parsed = parse_quran_pipe_file(raw_path)
raw_sha256 = sha256_file(raw_path)

print("Headers:", len(parsed["headers"]))
print("Records:", len(parsed["records"]))
print("Malformed:", len(parsed["malformed_lines"]))
print("Duplicates:", len(parsed["duplicate_keys"]))
print("SHA-256:", raw_sha256)

assert parsed["malformed_lines"] == []
assert parsed["duplicate_keys"] == []

<div dir="rtl">

# NB02-M04 — Canonical Numbering Validation

شماره آیات به‌جای اعتماد به تعداد کل، سوره‌به‌سوره بررسی می‌شود.

</div>

In [ ]:
# NB02-C04 — Validate canonical numbering
validation = validate_quran_records(
    parsed["records"],
    contract["expected_verse_counts_by_surah"],
)

print(json.dumps(validation, ensure_ascii=False, indent=2))
assert validation["status"] == "PASS"

<div dir="rtl">

# NB02-M05 — Source Evidence Binding

حقوق استفاده متن Tanzil برای نسخه verbatim ثبت شده است، اما Fingerprint فایل remote تاریخی در Git نگهداری نشده؛ این محدودیت صریح می‌ماند.

</div>

In [ ]:
# NB02-C05 — Evidence and header audit
header_text = "\n".join(parsed["headers"])
header_signals = {
    "mentions_tanzil": "tanzil" in header_text.lower(),
    "mentions_license": "license" in header_text.lower(),
    "header_count": len(parsed["headers"]),
}

assert (
    source_evidence["rights_interpretation"]["modification_of_quran_text"]
    == "not_allowed"
)

print(json.dumps(header_signals, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB02-M06 — Inventory Report

Report شامل Hash خام، تعداد و Structural checks است.

</div>

In [ ]:
# NB02-C06 — Persist inventory and acceptance report
reports_dir = project_root / "reports/phase1"
reports_dir.mkdir(parents=True, exist_ok=True)

inventory = {
    "schema_version": "qrag_quran_inventory_v1",
    "source_id": contract["source_id"],
    "raw_path": str(raw_path.relative_to(project_root)),
    "raw_sha256": raw_sha256,
    "raw_size_bytes": raw_path.stat().st_size,
    "header_signals": header_signals,
    "validation": validation,
    "source_evidence_status": source_evidence["evidence_status"],
    "limitations": source_evidence["limitations"],
}

inventory_path = reports_dir / "quran_canonical_inventory_v1.json"
write_json_atomic(inventory_path, inventory)

acceptance = {
    "schema_version": "qrag_nb02_acceptance_v1",
    "objective_status": "ACHIEVED",
    "acceptance_status": "PASS_WITH_LIMITATIONS",
    "failed_checks": [],
    "warnings": source_evidence["limitations"],
    "raw_sha256": raw_sha256,
    "record_count": len(parsed["records"]),
    "next_notebook": "notebooks/phase1/03_quran_canonical_ingestion_and_acceptance.ipynb",
}
acceptance_path = reports_dir / "quran_canonical_inventory_acceptance_v1.json"
write_json_atomic(acceptance_path, acceptance)

assert json.loads(inventory_path.read_text(encoding="utf-8")) == inventory
assert json.loads(acceptance_path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB02-M07 — جمع‌بندی

اگر این Notebook روی clone واقعی PASS شود، NB03 اجازه دارد نسخه canonical آیه‌محور را بدون تغییر متن تولید کند.

</div>